In [ ]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import f1_score

In [ ]:
train_data = pd.read_csv("data/train.csv")
train_data

,age,workclass,fnlwgt,education,education.num,marital.status,occupation,relationship,race,sex,capital.gain,capital.loss,hours.per.week,native.country,income
0,40,Self-emp-not-inc,223881,Prof-school,15,Married-civ-spouse,Prof-specialty,Husband,White,Male,99999,0,70,United-States,>50K
1,30,Private,149118,HS-grad,9,Divorced,Craft-repair,Not-in-family,White,Female,0,0,40,United-States,<=50K
2,46,Private,109209,Some-college,10,Married-civ-spouse,Adm-clerical,Husband,White,Male,0,0,40,United-States,>50K
3,32,Private,229566,Assoc-voc,11,Married-civ-spouse,Other-service,Husband,White,Male,0,0,60,United-States,>50K
4,54,?,148657,Preschool,1,Married-civ-spouse,?,Wife,White,Female,0,0,40,Mexico,<=50K
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
24995,40,Private,130834,Some-college,10,Never-married,Adm-clerical,Not-in-family,White,Female,0,0,40,United-States,<=50K
24996,31,Local-gov,33124,Bachelors,13,Never-married,Prof-specialty,Not-in-family,White,Female,0,0,50,United-States,<=50K
24997,38,Federal-gov,190895,Bachelors,13,Married-civ-spouse,Prof-specialty,Husband,White,Male,0,0,40,?,>50K
24998,23,Private,420973,Bachelors,13,Never-married,Prof-specialty,Not-in-family,White,Female,0,0,40,United-States,<=50K


In [ ]:
test_data = pd.read_csv("data/test.csv")
test_data

,age,workclass,fnlwgt,education,education.num,marital.status,occupation,relationship,race,sex,capital.gain,capital.loss,hours.per.week,native.country
0,31,Private,176711,HS-grad,9,Married-civ-spouse,Transport-moving,Husband,White,Male,0,0,38,United-States
1,40,Private,120277,Some-college,10,Never-married,Sales,Not-in-family,White,Male,0,0,40,United-States
2,43,Private,299197,Some-college,10,Married-civ-spouse,Adm-clerical,Husband,White,Male,0,0,40,United-States
3,70,Self-emp-inc,188260,HS-grad,9,Married-civ-spouse,Exec-managerial,Husband,White,Male,0,0,16,United-States
4,51,Private,254211,Bachelors,13,Married-civ-spouse,Craft-repair,Husband,White,Male,0,0,20,United-States
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
7159,37,Self-emp-not-inc,75050,Assoc-voc,11,Married-civ-spouse,Craft-repair,Husband,White,Male,0,0,55,United-States
7160,36,Private,156780,HS-grad,9,Never-married,Sales,Other-relative,Asian-Pac-Islander,Female,0,0,40,?
7161,34,Private,381153,HS-grad,9,Married-civ-spouse,Transport-moving,Husband,White,Male,0,0,40,United-States
7162,59,Private,294395,Masters,14,Widowed,Prof-specialty,Not-in-family,White,Female,0,0,40,United-States


In [ ]:
train_df = train_data.copy()
test_df  = test_data.copy()

train_df.replace("?", np.nan, inplace=True)
test_df.replace("?", np.nan, inplace=True)

for df in (train_df, test_df):
    for c in df.select_dtypes(include=["object"]).columns:
        df[c] = df[c].astype(str).str.strip()
        df.loc[df[c].isin(["nan", "None"]), c] = np.nan 


y = train_df["income"].astype(str).str.strip().str.replace(".", "", regex=False)
X = train_df.drop(columns=["income"])
X_test = test_df.copy()

num_cols = X.select_dtypes(exclude=["object"]).columns
cat_cols = X.select_dtypes(include=["object"]).columns

for c in num_cols:
    med = X[c].median()
    X[c] = X[c].fillna(med)
    if c in X_test.columns:
        X_test[c] = X_test[c].fillna(med)

for c in cat_cols:
    mode = X[c].mode(dropna=True)[0]
    X[c] = X[c].fillna(mode)
    if c in X_test.columns:
        X_test[c] = X_test[c].fillna(mode)

X_enc = pd.get_dummies(X, drop_first=False)
X_test_enc = pd.get_dummies(X_test, drop_first=False)

X_enc, X_test_enc = X_enc.align(X_test_enc, join="left", axis=1, fill_value=0)

X_tr, X_val, y_tr, y_val = train_test_split(
    X_enc, y, test_size=0.2, random_state=42, stratify=y
)

In [ ]:
model = RandomForestClassifier(
    n_estimators=1200,
    random_state=42,
    n_jobs=-1,
    max_features="sqrt",
    min_samples_leaf=2,
    min_samples_split=10,
    class_weight=None
)


model.fit(X_tr, y_tr)

RandomForestClassifier(min_samples_leaf=2, min_samples_split=10,
                       n_estimators=1200, n_jobs=-1, random_state=42)

In [ ]:
val_pred = model.predict(X_val)
f1 = f1_score(y_val, val_pred, pos_label=">50K")
print("Validation F1:", round(f1 * 100, 2))

Validation F1: 67.98


In [10]:
test_pred = model.predict(X_test_enc)
submission = pd.DataFrame({"income": test_pred})
submission.head()

,income
0,<=50K
1,<=50K
2,<=50K
3,<=50K
4,<=50K
